In [1]:
from polyconv_opt import fermion_logZ_numeric

In [2]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from joblib import Parallel, delayed
from physics import energy_nd as energy
from propagator import (p_funcs_zeta_1, p_funcs_lambda, p_funcs_k1,
                        factor_calc_T, factor_calc_H)
from plotting import make_tau_grid

def _compute_estimators_worker_fd_polyconv(args):
    tau_mid = args[0]
    tau_step = args[1]
    n = args[2]
    d = args[3]
    N = args[4]
    w = args[5]
    store_all_n = args[6] if len(args) > 6 else False
    
    tau_1 = tau_mid - 0.5 * tau_step
    tau_2 = tau_mid + 0.5 * tau_step
    
    eps_s = w * (tau_mid / N)
    lambda_val_s = p_funcs_lambda(eps_s)
    zeta_1_s = p_funcs_zeta_1(eps_s)
    k1_s = p_funcs_k1(eps_s)
    gamma_val_s = math.sqrt(max(0.0, zeta_1_s**2 - 1.0)) / k1_s
    
    fT_star = factor_calc_T_star(w, lambda_val_s, gamma_val_s)
    fH_star = factor_calc_H_star(w, lambda_val_s, gamma_val_s)
    
    if store_all_n:
        _, logZ1, _ = fermion_logZ_numeric(tau_1, N, n, d, w=w, return_all=True)
        _, logZ2, _ = fermion_logZ_numeric(tau_2, N, n, d, w=w, return_all=True)
        logZ1 = np.array(logZ1)
        logZ2 = np.array(logZ2)
        
        energy_T_all = (logZ1 - logZ2) / tau_step
        energy_H_all = energy_T_all * (fH_star / fT_star)
        
        return energy_T_all[1:], energy_H_all[1:]
    else:
        lq1 = fermion_logZ_numeric(tau_1, N, n, d, w=w)
        lq2 = fermion_logZ_numeric(tau_2, N, n, d, w=w)
        
        energy_T = (lq1 - lq2) / tau_step
        energy_H = energy_T * (fH_star / fT_star)
        
        return energy_T, energy_H

def plot_polyconv_vs_tau_dual(n, d, N_list, w, tau_start, tau_end, tau_step, save_filename="plot_data", store_all_n=False):
    taus_left = np.array(make_tau_grid(tau_start, tau_end, tau_step))
    taus_left = taus_left[taus_left + tau_step <= tau_end + 1e-15]
    taus_mid = taus_left + 0.5 * tau_step
    
    plt.figure(figsize=(10, 6))
    
    results_T = {"tau_mid": taus_mid}
    results_H = {"tau_mid": taus_mid}
    
    results_T_all_n = []
    results_H_all_n = []
    
    for N in N_list:
        tasks = [(tau, tau_step, n, d, N, w, store_all_n) for tau in taus_mid]
        results = Parallel(n_jobs=-1)(delayed(_compute_estimators_worker_fd_polyconv)(task) for task in tasks)
        
        if store_all_n:
            energies_T = np.array([res[0] for res in results])
            energies_H = np.array([res[1] for res in results])
            results_T_all_n.append(energies_T)
            results_H_all_n.append(energies_H)
            
            line, = plt.plot(taus_mid, energies_T[:, -1], label=f"N={N} (Thermo)")
            plt.plot(taus_mid, energies_H[:, -1], linestyle="--", color=line.get_color(), label=f"N={N} (Ham)")
        else:
            energies_T = np.array([res[0] for res in results])
            energies_H = np.array([res[1] for res in results])
            results_T[f"N_{N}"] = energies_T
            results_H[f"N_{N}"] = energies_H
            
            line, = plt.plot(taus_mid, energies_T, label=f"N={N} (Thermo)")
            plt.plot(taus_mid, energies_H, linestyle="--", color=line.get_color(), label=f"N={N} (Ham)")

    try:
        e = energy(n, d, w)
        print(f"Exact energy for n={n}, d={d}, w={w}: {e}")
        plt.axhline(e, linestyle=':', color='black', label=f"True Energy (n={n})")
    except NameError:
        pass 

    plt.xlabel(r"$\tau + \frac{1}{2}\,\Delta\tau$")
    plt.ylabel(r"Energy Estimators")
    plt.title(f"Thermodynamic (Solid) & Hamiltonian (Dashed) via Finite Difference\n(n={n}, d={d}, w={w}, PA Propagator)")
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    
    try:
        e = energy(n, d, w)
        plt.ylim(e*0.95, e*1.05) 
    except:
        pass
        
    plt.grid(True)
    plt.tight_layout()
    plt.show()
    
    if store_all_n:
        arr_T = np.array(results_T_all_n).transpose(2, 0, 1)
        arr_H = np.array(results_H_all_n).transpose(2, 0, 1)
        
        np.save(f"{save_filename}_T_all_n.npy", arr_T)
        np.save(f"{save_filename}_H_all_n.npy", arr_H)
        
        np.save(f"{save_filename}_taus.npy", taus_mid)
        np.save(f"{save_filename}_beads.npy", np.array(N_list))
        
        print(f"Data successfully saved to '{save_filename}_T_all_n.npy' (shape {arr_T.shape}) and '{save_filename}_H_all_n.npy' (shape {arr_H.shape})")
        print(f"Axes mapping: n (axis 0), beads (axis 1) -> '{save_filename}_beads.npy', taus (axis 2) -> '{save_filename}_taus.npy'")
        return arr_T, arr_H
    else:
        df_T = pd.DataFrame(results_T)
        df_H = pd.DataFrame(results_H)
        
        df_T.to_csv(f"{save_filename}_T.csv", index=False)
        df_H.to_csv(f"{save_filename}_H.csv", index=False)
        
        print(f"Data successfully saved to '{save_filename}_T.csv' and '{save_filename}_H.csv'")
        
        return df_T, df_H


In [ ]:
n_val = 100
w_val = 1.0

results = plot_polyconv_vs_tau_dual(
    n=n_val,
    d=2,
    N_list=[4],
    w=w_val,
    tau_start=0.5,
    tau_end=10.5,
    tau_step=0.5,
    save_filename=f"Saved_runs_and_plots/plot_data_n{n_val}_w{str(w_val).replace('.', '_')}",
    store_all_n=True
)


NameError: name 'factor_calc_T_star' is not defined

<Figure size 1000x600 with 0 Axes>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

n_val = 300
w_val = 1.0
save_filename = f"Saved_runs_and_plots/plot_data_n{n_val}_w{str(w_val).replace('.', '_')}"

try:
    arr_T = np.load(f"{save_filename}_T_all_n.npy")
    arr_H = np.load(f"{save_filename}_H_all_n.npy")
    taus = np.load(f"{save_filename}_taus.npy")
    beads = np.load(f"{save_filename}_beads.npy")

    print("Loaded data successfully.")
    print("arr_T shape:", arr_T.shape, "(n, beads, taus)")
    
    n_plot = 99 #index, n is +1
    bead_idx = 1
    
    if n_plot < arr_T.shape[0]:
        plt.figure(figsize=(8, 5))
        plt.plot(taus, arr_T[n_plot, bead_idx, :], label=f"T_estimator (N={beads[bead_idx]})")
        plt.plot(taus, arr_H[n_plot, bead_idx, :], label=f"H_estimator (N={beads[bead_idx]})", linestyle="--")
        plt.xlabel("tau_mid")
        plt.ylabel("Energy")
        plt.title(f"Extracted Estimators for n={n_plot+1}")
        plt.legend()
        plt.grid()
        plt.show()
    else:
        print(f"Requested n_plot={n_plot} is out of bounds for saved n_max={arr_T.shape[0]-1}")
        
except FileNotFoundError:
    print("Saved files not found. Run the previous cell with store_all_n=True first.")
